In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.
The file _gams_py_gdb2.gdx is still active and was not deleted.
The file _gams_py_gdb3.gdx is still active and was not deleted.
The file _gams_py_gdb4.gdx is still active and was not deleted.
The file _gams_py_gdb5.gdx is still active and was not deleted.


*Load specific modules used here:*

In [2]:
os.chdir(main)
%run stdPlotting.ipynb
os.chdir(d['py'])
import mCGE

# Load and run calibrated model

We can use the calibrated in one of two ways: By loading the model instance itself or by loading data and GAMS code. 

## 1. Run from model instance

*Load model:*

In [3]:
t0 = 2019
name = f'vGRS{t0}CGE'
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
db0 = M.db.copy() # store initial solution

### 1.1. Shocks: Increase CET parameter on recycled metal from "other"

Adjust data:

In [4]:
newMu = adj.rc_pd(M.db('mu'), pd.MultiIndex.from_tuples([('Other','Metal_R')], names = ['s','n'])) * 2
M.db.aom(newMu, name = 'mu', priority = 'second')

Run "simple" model:

In [5]:
M.db.mergeInternal() # write .gdx file based on Python database with updated taxes.
sol = M.solve(state = 'B') # Solve model in baseline state 'B' (as opposed to calibration state 'C')

#### A. Report some key statistics:

In [6]:
vm = adj.rc_pd(M.db('n2m2k2o'), pd.Index(['V'], name = 'k')).get_level_values('n')
rm = adj.rc_pd(M.db('n2m2k2o'), pd.Index(['R'], name = 'k')).get_level_values('n')
n2m = M.db('n2m2k2o').droplevel(['k','o'])

Reporting function:

In [7]:
def fReport(db_i):
    d = {}
    # virgin materials
    qD_vm = adj.rc_pd(db_i('qD'), vm).groupby(['t','n']).sum()
    d['qD_vm'] = adjMultiIndex.applyMult(qD_vm, n2m).groupby(['t','m']).sum()
    # recycled materials:
    qD_rm = adj.rc_pd(db_i('qD'), rm).groupby(['t','n']).sum()
    d['qD_rm'] = adjMultiIndex.applyMult(qD_rm, n2m).groupby(['t','m']).sum()
    # Total resource demand:
    d['qD_m'] = d['qD_vm'].add(d['qD_rm'], fill_value=0)
    # Economic value of all domestic outputs:
    d['totalValue'] = (db_i('qS') * db_i('p')).dropna().groupby('t').sum()
    # Share of materials from viring:
    d['Q_v/Q'] = d['qD_vm'].div(d['qD_m'])
    return d
def fCompare(d, d0):
    # Add to d
    d['ΔQ_v/Q_v'] = ((d['qD_vm']-d0['qD_vm'])/d0['qD_vm']) # relative change in use of virtual materials
    d['ΔQ_r/Q_r'] = ((d['qD_rm']-d0['qD_rm'])/d0['qD_rm']) # relative change in use of recycled materials
    d['Δ(Q_v/Q)/(Q_v/Q)'] = ((d['Q_v/Q']-d0['Q_v/Q'])/d0['Q_v/Q']) # relative change in the share of virgin materials out of all material throughput 
    return d
def fReportAndCompare(db_i, d0):
    d = fReport(db_i)
    d = fCompare(d, d0)
    return d

Get:

In [8]:
report0 = fReport(db0)
reportNewMu = fReportAndCompare(sol, report0)

### 1.2. Increase $\underline{\alpha}$:

Revert to initial data:

In [9]:
M.db.aom(db0('mu'), name = 'mu', priority = 'second')

Increase $\underline{\alpha}$ in recycling technoogy for metal:

In [10]:
new_αl = adj.rc_pd(M.db('WTD_alphal'), pd.Index(['Metal'], name = 'm'))+0.25
M.db.aom(new_αl, name = 'WTD_alphal', priority = 'second')
M.db.mergeInternal()
sol = M.solve(state='B')

Get:

In [11]:
reportNewAlpha = fReportAndCompare(sol, report0)

### 1.3. Add tax on virgin material inputs:

Revert to initial data:

In [12]:
M.db.aom(db0('WTD_alphal'), name = 'WTD_alphal', priority = 'second')

Add tax on virgin material inputs

In [13]:
M.db.aom(adj.rc_pd(M.db('tauD'), vm) + 0.25, name = 'tauD', priority = 'second')
M.db.mergeInternal()
sol = M.solve(state='B')

Get:

In [14]:
reportNewTax = fReportAndCompare(sol, report0)

### 1.4. Improve efficiency in service/material nest:

Revert to initial data:

In [15]:
M.db.aom(db0('tauD'), name = 'tauD', priority = 'second')

Get relevant share parameters: In the nesting structure this is the nest from ```RxE``` to ```ZY``` (services) and ```ZO``` (materials): 

In [16]:
newMu = adj.rc_pd(M.db('mu'), ('and', [pd.Index(['RxE'], name = 'n'), pd.Index(['ZO'], name = 'nn')])) * 1.25
M.db.aom(newMu, name = 'mu', priority = 'second')

Run:

In [17]:
M.db.mergeInternal() # write .gdx file based on Python database with updated taxes.
sol = M.solve(state = 'B') # Solve model in baseline state 'B' (as opposed to calibration state 'C')

Get:

In [18]:
reportNewTax = fReportAndCompare(sol, report0)